# RUNNING PINECONE INGESTION HERE.
- My Laptop's CPU crashed running it locally

In [1]:
!pip install -qU langchain langchain-huggingface langchain-text-splitters langchain-pinecone pinecone-client sentence-transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.


Get pinecone api key from colab secrets

In [2]:
from google.colab import userdata
pinecone_api_key = userdata.get('PINECONE_API_KEY')

In [3]:
import os
os.environ["PINECONE_API_KEY"] = pinecone_api_key

In [4]:
drive_folder = "/content/drive/MyDrive/Textbooks_Paeds_and_OnG_Markdown"

raw_documents = []
for filename in os.listdir(drive_folder):
  if filename.endswith(".md"):
    filepath = os.path.join(drive_folder, filename)

    with open(filepath, "r", encoding="utf-8") as file:
      content = file.read()

      raw_documents.append(
          {
          "text": content,
          "source": filename
          }
      )
print(f"✅ Successfully loaded {len(raw_documents)} Markdown files from Drive.")

✅ Successfully loaded 4 Markdown files from Drive.


In [ ]:
# Had issues with colab's installed dependencies, so had to do these
#I think this can be done in the first code cell instead

# 1. Uninstall the clashing libraries completely
!pip uninstall -y numpy scipy pandas langchain langchain-text-splitters

# 2. Reinstall them locked to stable, compatible versions
!pip install "numpy==1.26.4" "scipy==1.12.0" "pandas==2.2.1" langchain langchain-text-splitters langchain-huggingface langchain-pinecone sentence-transformers -q

print("✅ Environment harmonized. Ready for restart.")

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: langchain 1.2.10
Uninstalling langchain-1.2.10:
  Successfully uninstalled langchain-1.2.10
Found existing installation: langchain-text-splitters 1.1.1
Uninstalling langchain-text-splitters-1.1.1:
  Successfully uninstalled langchain-text-splitters-1.1.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account

In [5]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
     ("##", "Header 2"),
      ("###", "Header 3")
      ]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

final_chunks = []
for doc in raw_documents:
  header_splits = markdown_splitter.split_text(doc["text"])
  text_splits = text_splitter.split_documents(header_splits)

  for split in text_splits:
    split.metadata["source"] = doc["source"]

  final_chunks.extend(text_splits)

print(f"✅ Sliced the textbooks into {len(final_chunks)} RAG-ready chunks.")

✅ Sliced the textbooks into 12506 RAG-ready chunks.


In [ ]:
!pip install -q sentence-transformers

In [6]:
# CELL 4
from langchain_huggingface import HuggingFaceEmbeddings

print("Downloading BGE-Large and loading directly into T4 GPU VRAM...")

# The critical change: 'device': 'cuda' routes the math to the NVIDIA GPU
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

print("✅ GPU Embedding Engine Online!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ GPU Embedding Engine Online!


In [7]:

from tqdm import tqdm
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from google.colab import userdata

# 1. Securely fetch your Pinecone API Key
pinecone_api_key = userdata.get('PINECONE_API_KEY')
index_name = "mb-assistant"

print(f"🔌 Connecting to Pinecone Index: {index_name}...")
pc = Pinecone(api_key=pinecone_api_key)

# Connect LangChain to the empty index
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# 2. Fire the chunks to Pinecone
batch_size = 100
total_chunks = len(final_chunks)

print(f"\n🚀 Beginning high-speed GPU ingestion of {total_chunks} chunks...")

for i in tqdm(range(0, total_chunks, batch_size), desc="Embedding & Uploading"):
    batch = final_chunks[i : i + batch_size]
    vectorstore.add_documents(documents=batch)

print("\n🎉 INGESTION COMPLETE! Your RAG Brain is officially online.")

🔌 Connecting to Pinecone Index: mb-assistant...

🚀 Beginning high-speed GPU ingestion of 12506 chunks...


Embedding & Uploading: 100%|██████████| 126/126 [09:27<00:00,  4.50s/it]


🎉 INGESTION COMPLETE! Your RAG Brain is officially online.
